## Ejemplo de Entrenamiento de una Red Neuronal con Pytorch

In [11]:
import sys

if "google.colab" in sys.modules:
    %pip install -q torchmetrics torchinfo

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)

## Separación de los Datos en Train y Validación

In [ ]:
from sklearn.model_selection import train_test_split

X = data.data
y = data.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

In [ ]:
from torch.utils.data import Dataset


class WineDataset(Dataset):
    def __init__(self, X, y):
        ## Convertimos a numpy arrays
        self.X = X.to_numpy()
        self.y = y.values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return dict(
            ## Tranformarmos cada índice en Tensor de Floats.
            X=torch.from_numpy(self.X)[idx].float(),
            y=torch.from_numpy(self.y)[idx].long(),
        )


train_data = WineDataset(X_train, y_train)
val_data = WineDataset(X_val, y_val)

## DataLoader

In [ ]:
from torch.utils.data import DataLoader

bs = 128
train_loader = DataLoader(
    train_data,
    batch_size=bs,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_data,
    batch_size=bs,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

In [ ]:
from torchinfo import summary


class WineNeuralNet(nn.Module):
    def __init__(self, n_features=13, n_hidden=8, n_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(n_features, n_hidden)
        self.relu = nn.ReLU(inplace=True)
        # self.fc2 = nn.Linear(n_hidden, n_hidden)
        # self.relu_2 = nn.ReLU(inplace=True)
        self.fc3 = nn.Linear(n_hidden, n_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        # x = self.fc2(x)
        # x = self.relu_2(x)
        x = self.fc3(x)
        return x


model = WineNeuralNet(n_features=13, n_hidden=8, n_classes=3)
summary(model, input_size=(32, 13))
## Probar con 8 hidden Dims, 24, y luego agregando algunas capas.

## Entrenamiento del Modelo

In [ ]:
import numpy as np
from torchinfo import summary
from torchmetrics.classification import MulticlassAccuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
## Probar Entrenamiento con y sin reiniciar el Modelo.

epochs = 200  # probar con 500
model = WineNeuralNet(n_features=13, n_hidden=8, n_classes=3)
model.to(device)
summary(model, input_size=(32, 13))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
epoch_loss = dict(train=[], val=[])
epoch_metric = dict(train=[], val=[])
for e in range(epochs):
    train_metric, val_metric = (
        MulticlassAccuracy(num_classes=3).to(device),
        MulticlassAccuracy(num_classes=3).to(device),
    )
    batch_loss = dict(train=[], val=[])

    model.train()
    for batch in train_loader:
        X, y = batch["X"].to(device), batch["y"].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        acc = train_metric(logits, y)
        optimizer.step()
        batch_loss["train"].append(loss.item())

    tr_acc = train_metric.compute()
    train_epoch_loss = np.mean(batch_loss["train"])
    epoch_metric["train"].append(tr_acc.item())

    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            X, y = batch["X"].to(device), batch["y"].to(device)
            logits = model(X)
            loss = criterion(logits, y)
            batch_loss["val"].append(loss.item())
            acc = val_metric(logits, y)

        val_acc = val_metric.compute()
        val_epoch_loss = np.mean(batch_loss["val"])
        epoch_metric["val"].append(val_acc.item())

    epoch_loss["train"].append(train_epoch_loss)
    epoch_loss["val"].append(val_epoch_loss)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(epoch_loss["train"], label="Train Loss")
plt.plot(epoch_loss["val"], label="Validation Loss")
plt.title("Loss vs Epochs")
plt.legend()
plt.show()


In [ ]:
plt.plot(epoch_metric["train"], label="Train Accuracy")
plt.plot(epoch_metric["val"], label="Validation Accuracy")
plt.title("Accuracy vs Epochs")
plt.legend()
plt.show()